In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import re
from urllib.parse import urljoin
from datetime import datetime
import os

def scrape_nrb_faqs():
    base_url = "https://www.nrb.org.np"
    start_url = "https://www.nrb.org.np/category/faqs/"
    
    # Use session for connection pooling
    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    })
    
    all_faqs = []
    
    def get_page_content(url):
        """Fetch and parse page content"""
        try:
            print(f"Fetching: {url}")
            response = session.get(url, timeout=30)
            response.raise_for_status()
            return BeautifulSoup(response.content, "html.parser")
        except Exception as e:
            print(f"Error fetching {url}: {e}")
            return None
    
    def extract_post_count(text):
        """Extract number from text like '7 categories' or '2 posts'"""
        if not text:
            return 0
        match = re.search(r'(\d+)', text)
        return int(match.group(1)) if match else 0
    
    def process_faq_page(soup, category_name=""):
        """Process FAQ page to extract questions and answers from collapse divs"""
        page_faqs = []
        
        # Find all buttons with data-toggle="collapse"
        buttons = soup.find_all('button', {'data-toggle': 'collapse'})
        
        print(f"    Found {len(buttons)} FAQ buttons")
        
        for button in buttons:
            try:
                # Extract question from button text
                question = button.get_text(strip=True)
                # Remove the chevron icon text if present
                question = re.sub(r'\s*$', '', question)
                
                # Get the target collapse div ID
                target = button.get('data-target', '')
                if target.startswith('#'):
                    collapse_id = target[1:]  # Remove the #
                    
                    # Find the corresponding collapse div
                    collapse_div = soup.find('div', {'id': collapse_id})
                    
                    if collapse_div:
                        # Find the card-body inside the collapse div
                        card_body = collapse_div.find('div', class_='card-body')
                        
                        if card_body:
                            # Extract answer text
                            answer = card_body.get_text(separator=' ', strip=True)
                            print(answer)
                            
                            if question and answer:
                                page_faqs.append({
                                    'category': category_name,
                                    'question': question,
                                    'answer': answer
                                })
                                print(f"      ✓ Extracted FAQ: {question[:50]}...")
                        else:
                            print(f"      ⚠ No card-body found for: {question[:50]}...")
                    else:
                        print(f"      ⚠ No collapse div found with id: {collapse_id}")
                        
            except Exception as e:
                print(f"      Error processing button: {e}")
                continue
        
        return page_faqs
    
    def scrape_category_pages(category_url, category_name):
        """Scrape all pages of a category"""
        category_faqs = []
        page_num = 1
        
        while category_url:
            print(f"  Processing page {page_num} of {category_name}")
            
            soup = get_page_content(category_url)
            if not soup:
                break
            
            # Process current page
            page_faqs = process_faq_page(soup, category_name)
            category_faqs.extend(page_faqs)
            print(f"    Collected {len(page_faqs)} FAQs from this page")
            
            # Look for next page
            next_page = None
            pagination_div = soup.find('div', class_='pagination-block')
            if pagination_div:
                # Find all pagination links
                page_links = pagination_div.find_all('a')
                for link in page_links:
                    link_text = link.get_text(strip=True)
                    if link_text and link_text.isdigit():
                        if int(link_text) == page_num + 1:
                            next_page = urljoin(base_url, link['href'])
                            break
                    elif 'Next' in link_text or '»' in link_text or 'next' in link_text.lower():
                        next_page = urljoin(base_url, link['href'])
            
            category_url = next_page
            page_num += 1
            
            if category_url:
                time.sleep(1)  # Polite delay
        
        return category_faqs
    
    def save_to_text_file(faqs_list, filename):
        """Save FAQs to a nicely formatted text file"""
        try:
            filepath = os.path.join(os.getcwd(), filename)
            print(f"\nAttempting to save to: {filepath}")
            
            with open(filepath, 'w', encoding='utf-8') as f:
                f.write("=" * 80 + "\n")
                f.write("NRB FAQ SCRAPER RESULTS\n")
                f.write(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                f.write(f"Total FAQs: {len(faqs_list)}\n")
                f.write("=" * 80 + "\n\n")
                
                current_category = ""
                faq_counter = 1
                
                for faq in faqs_list:
                    if faq['category'] != current_category:
                        current_category = faq['category']
                        f.write("\n" + "=" * 80 + "\n")
                        f.write(f"CATEGORY: {current_category}\n")
                        f.write("=" * 80 + "\n\n")
                        faq_counter = 1
                    
                    f.write(f"FAQ #{faq_counter}\n")
                    f.write("-" * 40 + "\n")
                    f.write(f"Q: {faq['question']}\n")
                    f.write(f"A: {faq['answer']}\n")
                    f.write("\n")
                    faq_counter += 1
            
            print(f"✓ Successfully saved to: {filepath}")
            return filepath
            
        except Exception as e:
            print(f"❌ ERROR saving file: {e}")
            return None
    
    # Main scraping logic
    print("Starting NRB FAQ Scraper...")
    print("=" * 50)
    
    # Get initial page
    html = get_page_content(start_url)
    if not html:
        print("Failed to fetch initial page!")
        return []
    
    # Find the main body
    body = html.find("div", class_="card-bt-body")
    if not body:
        print("Could not find card-bt-body!")
        return []
    
    # Get all top-level categories
    categories = body.find_all("div", class_="col-12 col-lg-3 col-sm-6 col-md-4")
    print(f"Found {len(categories)} categories on main page\n")
    
    all_faqs = []
    
    for div in categories:
        try:
            # Try to find card-text
            card_text = div.find(['p', 'div'], class_='card-text')
            if not card_text:
                continue
            
            span = card_text.find('span')
            if not span:
                continue
            
            span_text = span.get_text(strip=True)
            post_count = extract_post_count(span_text)
            
            if post_count > 0:
                anchor = div.find("a", href=True)
                if not anchor:
                    continue
                
                category_name = anchor.get_text(strip=True)
                link = urljoin(base_url, anchor['href'])
                
                print(f"Processing category: {category_name} ({post_count} items)")
                
                # Fetch category page
                response = session.get(link, timeout=30)
                category_html = BeautifulSoup(response.content, "html.parser")
                
                # Check if this is a FAQ page or another category page
                category_body = category_html.find("div", class_="card-bt-body")
                if category_body:
                    # Check for subcategories
                    subcategories = category_body.find_all("div", class_="col-12 col-lg-3 col-sm-6 col-md-4")
                    
                    if subcategories:
                        # This is another level of categories
                        print(f"  Found {len(subcategories)} subcategories")
                        for sub_div in subcategories:
                            sub_card_text = sub_div.find(['p', 'div'], class_='card-text')
                            if sub_card_text:
                                sub_span = sub_card_text.find('span')
                                if sub_span:
                                    sub_span_text = sub_span.get_text(strip=True)
                                    sub_post_count = extract_post_count(sub_span_text)
                                    
                                    if sub_post_count > 0:
                                        sub_anchor = sub_div.find("a", href=True)
                                        if sub_anchor:
                                            sub_name = sub_anchor.get_text(strip=True)
                                            sub_link = urljoin(base_url, sub_anchor['href'])
                                            
                                            print(f"  Processing subcategory: {sub_name} ({sub_post_count} items)")
                                            
                                            # Scrape this subcategory
                                            sub_faqs = scrape_category_pages(sub_link, f"{category_name} > {sub_name}")
                                            all_faqs.extend(sub_faqs)
                                            
                                            time.sleep(1)
                    else:
                        # This is a FAQ page
                        faqs = scrape_category_pages(link, category_name)
                        all_faqs.extend(faqs)
                
                time.sleep(1)
                
        except Exception as e:
            print(f"Error processing category: {e}")
            continue
    
    # Save results to text file
    if all_faqs:
        # Generate filename
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f'nrb_faqs_{timestamp}.txt'
        
        # Save to text file
        save_to_text_file(all_faqs, filename)
        
        print(f"\n{'='*50}")
        print(f"Scraping completed!")
        print(f"Total FAQs collected: {len(all_faqs)}")
        print(f"Data saved to: {filename}")
        
        # Show sample
        if all_faqs:
            print("\nFirst 3 FAQs from output:")
            print("-" * 50)
            for i, faq in enumerate(all_faqs[:3], 1):
                print(f"\n{i}. Category: {faq['category']}")
                print(f"   Question: {faq['question'][:80]}...")
                print(f"   Answer: {faq['answer'][:80]}...")
    else:
        print("\n❌ No FAQs were collected!")
    
    return all_faqs

if __name__ == "__main__":
    scrape_nrb_faqs()